# Minimum Vertex Cover
(see also [Wikipedia](https://en.wikipedia.org/wiki/Vertex_cover))

A vertex cover of a graph $G$ is a set of nodes such that every edge has at least one end in the set. To compare with the [PennyLane implementation](https://github.com/PennyLaneAI/pennylane/blob/master/pennylane/qaoa/cost.py), we first build its objective by hand. With $x_i = 1$ for the chosen nodes and spins $\sigma_i = 2x_i - 1$, it reads

$$
    C = \frac 34 \sum_{(i, j) \in E(G)} (\sigma_i \sigma_j - \sigma_i - \sigma_j) + \sum_{i \in V(G)} \sigma_i,
$$

where $E(G)$ is the set of edges and $V(G)$ is the set of nodes of $G$. Its minima are the minimum vertex covers.

In [ ]:
using QAOA

using PyPlot
PyPlot.plt.style.use("paper.mplstyle")
using PyCall
nx = pyimport("networkx");

__Defining the problem by hand:__

The following graph should be the same as [in this example](https://pennylane.ai/qml/demos/tutorial_qaoa_intro.html):

In [ ]:
N = 4
graph = nx.gnp_random_graph(N, 0.5, seed=7)

figure(figsize=(3, 2))
nx.draw(graph, with_labels=true)

Note that we have to __shift the edges by 1__ when going from Python to Julia:

In [ ]:
edges = [edge .+ (1, 1) for edge in graph.edges]
h = ones(N)
J = zeros(N, N)
for (i, j) in edges
    h[i] -= 3/4
    h[j] -= 3/4
    J[i, j] = J[j, i] = 3/4
end
mvc_problem = IsingProblem(h, J)    # minimized by default

__Using the wrapper function:__ `min_vertex_cover` uses a QUBO instead, $\sum_i x_i + P \sum_{(i, j) \in E(G)} (1 - x_i)(1 - x_j)$, with the penalty $P = 2$ for every edge without a chosen end. It has the same minima.

In [ ]:
mvc_problem = min_vertex_cover(N, edges)

__Optimizing the angles:__ `optimize_schedule` tunes the angles of `p` layers and returns them with the average objective they give:

In [ ]:
p = 2
angles, value = optimize_schedule(mvc_problem, p)

In [ ]:
probs = probabilities(mvc_problem, angles)

xlabels = []
for bstr in digits.(0:2^N-1, base=2, pad=N)
    push!(xlabels, "\$|" * prod([string(b) for b in bstr]) * "\\rangle\$")
end

figure(figsize=(5, 3.2))
ax = subplot(111)
bar(0:2^N-1, probs)
ax.set_xticks(0:2^N-1)
ax.set_xticklabels(xlabels, rotation=90)
minorticks_off()
tight_layout()

The states `5` $\equiv |1010\rangle$ and `6` $\equiv |0110\rangle$ are indeed the correct solutions!

### The XY mixer

The `XYMixer` moves amplitude only between assignments with the same number of ones. Started in `dicke_state(N, k)`, the equal superposition of all assignments with $k$ ones, the QAOA searches among the sets of exactly $k$ nodes. Both minimum covers have two nodes:

In [ ]:
mixer = XYMixer([(1, 2), (2, 3), (3, 4), (4, 1)])
ws = Workspace(mvc_problem; mixer, initial_state = dicke_state(N, 2))
angles, value = optimize_schedule(ws, p)
probs = probabilities(ws, angles)

All amplitude stays on the six sets of two nodes, and the two covers are the most likely:

In [ ]:
xlabels = []
for bstr in digits.(0:2^N-1, base=2, pad=N)
    push!(xlabels, "\$|" * prod([string(b) for b in bstr]) * "\\rangle\$")
end

figure(figsize=(5, 3.2))
ax = subplot(111)
bar(0:2^N-1, probs)
ax.set_xticks(0:2^N-1)
ax.set_xticklabels(xlabels, rotation=90)
minorticks_off()
tight_layout()